# HTB SMS preprocessing: watch each text stage

Module 292, Section 11. Use the **AI** kernel. This follows the exact course order—lowercase, regex cleaning, tokenization, stop-word removal, stemming, rejoin—on the UCI SMS Spam Collection. Named intermediate tables preserve the original text so you can see what changes. No model or text vectorizer is introduced yet.

## Reuse the Section 10 archive and exact-deduplicated starting table

Set `SMS_SPAM_ZIP` to a local UCI ZIP if present; otherwise fetch the same course link. Read only the expected archive member in memory. `quoting=csv.QUOTE_NONE` prevents literal quote marks inside SMS text from merging two message lines. An exact duplicate has the same label **and** message. This setup is carried forward from the preceding section, not a new modeling step.

In [ ]:
from io import BytesIO
from pathlib import Path
from zipfile import ZipFile
import csv
import os
import re
import nltk
import pandas as pd
import requests

source_url = 'https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip'
local_zip = Path(os.environ.get('SMS_SPAM_ZIP', 'sms+spam+collection.zip'))
if local_zip.is_file():
    archive_bytes = local_zip.read_bytes()
else:
    response = requests.get(source_url, timeout=30)
    response.raise_for_status()
    archive_bytes = response.content
if len(archive_bytes) > 5_000_000:
    raise ValueError('Unexpectedly large SMS archive')
with ZipFile(BytesIO(archive_bytes)) as archive:
    member = archive.getinfo('SMSSpamCollection')
    if member.file_size > 2_000_000:
        raise ValueError('Unexpectedly large SMS text member')
    with archive.open(member) as source:
        raw_sms = pd.read_csv(source, sep='\t', header=None,
                              names=['label', 'message'], quoting=csv.QUOTE_NONE)
starting_sms = raw_sms.drop_duplicates().reset_index(drop=True)
assert starting_sms[['label', 'message']].notna().all().all()
assert starting_sms['label'].isin(['ham', 'spam']).all()
print('Rows after exact deduplication:', len(starting_sms))

## Check NLTK resources before processing

HTB downloads `punkt`, `punkt_tab`, and `stopwords`. `word_tokenize` needs the tokenizer data; `stopwords.words('english')` needs the English list. This cell checks local installation first, downloading only a missing resource. All three were already installed during validation in the **AI** environment. The download is a dependency step, not text cleaning itself.

In [ ]:
resources = {'tokenizers/punkt': 'punkt',
             'tokenizers/punkt_tab': 'punkt_tab',
             'corpora/stopwords': 'stopwords'}
for resource_path, package_name in resources.items():
    try:
        nltk.data.find(resource_path)
    except LookupError:
        if not nltk.download(package_name, quiet=True, raise_on_error=True):
            raise RuntimeError(f'Could not obtain NLTK resource: {package_name}')
print('NLTK tokenizer and English stop-word resources are ready')

## 1. Lowercase

`FREE` and `free` become one spelling, lowering vocabulary size. `starting_sms` stays unchanged; `lower_sms` is a copy. This operation is deterministic but cannot be undone without the original. We show a small preview rather than the entire corpus.

In [ ]:
lower_sms = starting_sms.copy()
lower_sms['message'] = lower_sms['message'].str.lower()
print(lower_sms[['label', 'message']].head(3))
assert lower_sms['label'].equals(starting_sms['label'])
assert lower_sms['message'].map(lambda text: text == text.lower()).all()

## 2. Keep letters, whitespace, `$`, and `!`

HTB's regex `[^a-z\s$!]` reads **any character not among lowercase a through z, whitespace, dollar sign, or exclamation point**. `re.sub(..., '', text)` deletes every such character. For example, `FREE $20!!!` becomes `free $!!!`: the number vanishes but the symbols survive. Removing digits can also erase phone numbers, prices, and phishing clues, so this is a course demonstration rather than a universal cleanup rule.

In [ ]:
chars_sms = lower_sms.copy()
chars_sms['message'] = chars_sms['message'].map(
    lambda text: re.sub(r'[^a-z\s$!]', '', text))
print(chars_sms[['label', 'message']].head(3))
assert re.sub(r'[^a-z\s$!]', '', 'free $20!!!') == 'free $!!!'
assert chars_sms['label'].equals(lower_sms['label'])

## 3. Tokenize into lists

`word_tokenize` turns each string into a list of smaller units. A unit may be punctuation such as `!`, not just a word. HTB keeps the exclamation mark because it may convey spammy emphasis. From this point through stemming, `message` contains lists rather than strings, so code expecting raw text would have the wrong type.

In [ ]:
from nltk.tokenize import word_tokenize

tokens_sms = chars_sms.copy()
tokens_sms['message'] = tokens_sms['message'].map(word_tokenize)
print(tokens_sms[['label', 'message']].head(3))
assert tokens_sms['message'].map(lambda value: isinstance(value, list)).all()
assert word_tokenize('free prize!') == ['free', 'prize', '!']

## 4. Remove common stop words

A stop-word set contains frequent English items such as `the` and `and`. The list comprehension retains a token only if it is **not** in the set. This saves feature space but can lose negation or other context; check whether `not` appears in the installed set rather than assuming it is harmless to remove. Symbols retained above are not ordinary English stop words.

In [ ]:
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))
no_stop_sms = tokens_sms.copy()
no_stop_sms['message'] = no_stop_sms['message'].map(
    lambda tokens: [token for token in tokens if token not in stop_words])
print(no_stop_sms[['label', 'message']].head(3))
print('Is not a stop word in this NLTK list?', 'not' in stop_words)
assert no_stop_sms['message'].map(
    lambda tokens: all(token not in stop_words for token in tokens)).all()

## 5. Apply Porter stemming

A stemmer mechanically folds related spellings toward a common stem. `running` → `run` is a simple example; other outputs may be fragments, not dictionary words. This can reduce feature count but also merge distinctions, so whether it helps is an empirical question for a later held-out evaluation.

In [ ]:
from nltk.stem import PorterStemmer

stemmer = PorterStemmer()
stemmed_sms = no_stop_sms.copy()
stemmed_sms['message'] = stemmed_sms['message'].map(
    lambda tokens: [stemmer.stem(token) for token in tokens])
print(stemmed_sms[['label', 'message']].head(3))
assert stemmer.stem('running') == 'run'
assert stemmed_sms['message'].map(lambda value: isinstance(value, list)).all()

## 6. Join tokens back into strings

`' '.join(tokens)` places one space between the remaining tokens, producing the string format a later vectorizer can read. This does **not** restore removed digits, punctuation, stop words, or capitalization. Keep `starting_sms` for provenance. No text vectorization or model fit is performed in this section.

In [ ]:
processed_sms = stemmed_sms.copy()
processed_sms['message'] = processed_sms['message'].map(' '.join)
print(processed_sms[['label', 'message']].head(3))
assert processed_sms['message'].map(lambda value: isinstance(value, str)).all()
assert processed_sms['label'].equals(starting_sms['label'])
assert len(processed_sms) == len(starting_sms)
print('Rows preserved through all text stages:', len(processed_sms))

The section has produced normalized strings and preserved labels. A real spam filter must apply the *same* text pipeline at training and prediction time. HTB's character and stop-word choices can remove valuable phishing evidence, so their effect on precision/recall must eventually be tested rather than assumed beneficial. The next course material, not this notebook, handles turning strings into numeric model features.

# Section 12 — Feature Extraction

The cells above completed Section 11's preprocessing. This section starts with its `processed_sms` strings and creates a numeric **bag-of-words** matrix using HTB's exact `CountVectorizer(min_df=1, max_df=0.9, ngram_range=(1, 2))` settings. No classifier is trained here. Run this notebook top to bottom so `processed_sms` exists before these cells.

## What one matrix cell means

`X[i, j]` is read **X sub i comma j**: row `i` is one SMS message, column `j` is one learned vocabulary term, and the number is that term's count in that message. If the vocabulary is `[free, prize, free prize]`, then `free prize free` has counts `[2, 1, 1]`. A **unigram** is one term; a **bigram** is an adjacent pair. `ngram_range=(1, 2)` means include both. A unigram-only bag loses word order; bigrams retain just one neighboring relationship, not whole-sentence meaning. `min_df=1` keeps a term in at least one document; `max_df=0.9` removes terms found in more than 90% of documents. Document frequency counts *messages containing* a term, not repeats inside one message.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

# Five short messages from HTB's explanation make the column mapping visible.
toy_messages = [
    'The free prize is waiting for you',
    'The spam message offers a free prize now',
    'The spam filter might detect this',
    'The important news says you won a free trip',
    'The message truly is important',
]
toy_vectorizer = CountVectorizer(min_df=1, max_df=0.9, ngram_range=(1, 2))
toy_X = toy_vectorizer.fit_transform(toy_messages)
toy_terms = toy_vectorizer.get_feature_names_out()
print('Toy matrix shape (messages, terms):', toy_X.shape)
print('Selected toy columns:')
for term in ('free', 'prize', 'free prize'):
    column = toy_vectorizer.vocabulary_[term]
    print(term, toy_X[:, column].toarray().ravel().tolist())  # Only 5 rows.
# 'the' occurs in all five messages, so its document frequency 5/5 > 0.9.
assert 'the' not in toy_vectorizer.vocabulary_
# The default token pattern excludes one-character words and standalone symbols.
assert all(term not in toy_vectorizer.vocabulary_ for term in ('a', '$', '!'))
assert toy_X.shape[0] == len(toy_messages)
assert toy_X[0, toy_vectorizer.vocabulary_['free prize']] == 1

The tiny five-row example is the **only** matrix we partially densify for display. HTB's drawn table includes a column for `a`, but scikit-learn's default token pattern keeps tokens of at least two word characters, so that column is absent from actual `CountVectorizer` output. The same default also ignores standalone `$` and `!` symbols that Section 11 intentionally retained; preserving them in cleaned text does **not** make them model features under these vectorizer settings. `the` appears in all five documents and is removed by `max_df=0.9`; some *bigrams* containing `the` can still survive because each bigram has its own document frequency. The real corpus below stays sparse to avoid a large, mostly-zero dense allocation.

## Apply HTB's settings to the preprocessed SMS corpus

`fit_transform` learns the vocabulary and fills a **sparse** matrix of term counts. Sparse storage records only nonzero cells, which is appropriate when most terms are absent from most messages. `X.shape` is `(number of messages, number of vocabulary features)`; `X.nnz` is the number of nonzero cells. `y` contains the answers in the same row order: `1` for spam and `0` for ham. We verify that correspondence and do **not** convert the full matrix to a dense DataFrame. Fitting a vocabulary on the whole corpus follows this HTB demonstration; a later honest evaluation must fit it on training data only to avoid test-set vocabulary leakage.

In [ ]:
vectorizer = CountVectorizer(min_df=1, max_df=0.9, ngram_range=(1, 2))
X = vectorizer.fit_transform(processed_sms['message'])
y = processed_sms['label'].map({'ham': 0, 'spam': 1}).astype(int)
feature_names = vectorizer.get_feature_names_out()
print('Sparse count matrix shape:', X.shape)
print('Nonzero count cells:', X.nnz)
print('Vocabulary examples:', feature_names[:12].tolist())
print('Label counts:', y.value_counts().sort_index().to_dict())
assert X.shape[0] == len(processed_sms) == len(y)
assert X.shape[1] == len(feature_names) == len(vectorizer.vocabulary_)
assert X.nnz > 0 and X.nnz < X.shape[0] * X.shape[1]
assert set(y.unique()) == {0, 1}
assert (y.eq(1) == processed_sms['label'].eq('spam')).all()
assert X.format == 'csr'  # Compressed sparse row storage; do not call X.toarray().

Section 12 ends with numeric `X` and aligned binary labels `y`, ready for a **later** classifier lesson. These are raw term counts, not TF-IDF weights or calibrated probabilities. The full-corpus fit is HTB's illustration, not a leakage-safe evaluation pipeline. The matrix remains sparse; printing or converting the entire matrix to dense form would waste memory and could reproduce the Python memory problems we have encountered elsewhere.

# Section 13 — Training and Evaluation (Spam Detection)

The earlier sections left us with `processed_sms` strings and aligned labels `y`. HTB now combines `CountVectorizer` and `MultinomialNB` in a `Pipeline`, searches eight smoothing values with five-fold cross-validation, predicts five new messages, and demonstrates `joblib` serialization. **Evaluation correction:** HTB searches on all messages, leaving no untouched test set. We split first, tune only on training rows, and score the held-out test rows once. The Section 12 full-corpus `X`/`vectorizer` are intentionally *not* reused in this evaluation.

## What alpha smoothing does

For a word `w` and class `c`, a simplified Multinomial Naive Bayes estimate is `P(w | c) ≈ (N[w,c] + α) / (N[c] + αV)`. Read this **probability of word w given class c approximately equals the count of w in c plus alpha, divided by the total token count in c plus alpha times vocabulary size**. `α` (say **alpha**) is a nonnegative smoothing amount; `V` is the number of vocabulary terms. If w occurred 2 times among 10 class tokens, V=5, and alpha=1, the estimate is `(2+1)/(10+5)=0.20`; an unseen word gets `(0+1)/15`, not zero. HTB tests alpha values from 0.01 to 1.0. This is about learned token counts, **not** adding alpha to message text or labels.

## Reserve a test set before any vocabulary or alpha search

`train_test_split(..., stratify=y)` roughly preserves the ham/spam mix on both sides. The random seed makes this classroom split repeatable. The 20% test rows will not enter `GridSearchCV`; they are held until after model selection. `train_text` and `y_train` remain in the same row order. A real deployment might split by time or sender/campaign to avoid near-duplicate messages across folds; this lesson's random split does not prove that stronger generalization.

In [ ]:
from sklearn.model_selection import train_test_split

train_text, test_text, y_train, y_test = train_test_split(
    processed_sms['message'], y, test_size=0.20,
    stratify=y, random_state=1337)
print('Training rows:', len(train_text), 'Held-out test rows:', len(test_text))
print('Training labels:', y_train.value_counts().sort_index().to_dict())
print('Test labels:', y_test.value_counts().sort_index().to_dict())
assert len(train_text) + len(test_text) == len(processed_sms)
assert train_text.index.is_unique and test_text.index.is_unique
assert set(train_text.index).isdisjoint(test_text.index)
assert train_text.index.equals(y_train.index)
assert test_text.index.equals(y_test.index)

## Fit a pipeline and tune alpha using training rows only

A `Pipeline` first learns a word-count vocabulary and then fits a `MultinomialNB` classifier. `GridSearchCV` makes five train/validation folds within the training rows and compares HTB's eight alpha values (up to 8 × 5 = 40 fold fits, plus a final refit). `scoring='f1'` ranks settings by the harmonic mean of spam precision and recall. `n_jobs=1` keeps one fit active at a time to limit memory pressure. A **fresh** vectorizer is placed in this pipeline; the earlier full-corpus demonstration vectorizer is not used. The resulting `best_model` includes both stages.

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer

pipeline = Pipeline([
    ('vectorizer', CountVectorizer(min_df=1, max_df=0.9, ngram_range=(1, 2))),
    ('classifier', MultinomialNB()),
])
alpha_values = [0.01, 0.1, 0.15, 0.2, 0.25, 0.5, 0.75, 1.0]
grid_search = GridSearchCV(
    pipeline, {'classifier__alpha': alpha_values}, cv=5,
    scoring='f1', n_jobs=1, error_score='raise')
grid_search.fit(train_text, y_train)  # Never fit on test_text.
best_model = grid_search.best_estimator_
print('Best alpha:', grid_search.best_params_['classifier__alpha'])
print('Mean cross-validation F1:', round(grid_search.best_score_, 4))
assert grid_search.best_params_['classifier__alpha'] in alpha_values
assert best_model.named_steps['vectorizer'].get_feature_names_out().size > 0

## Evaluate once on the untouched test set

`best_model.predict(test_text)` applies the *training-fitted* vectorizer and classifier in one call. In the confusion matrix, rows are true labels and columns are predicted labels; with `[0,1]` ordering, cells are TN, FP, FN, TP. Say **true negative, false positive, false negative, true positive**. F1, precision, and recall have the meanings from Section 8. These test results are evidence for this one random split, not a guarantee about real-world phishing. Do not tune alpha again after viewing the test scores; that would make the test set part of model selection.

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

test_predictions = best_model.predict(test_text)
matrix = confusion_matrix(y_test, test_predictions, labels=[0, 1])
tn, fp, fn, tp = matrix.ravel()
print('Confusion matrix [[TN, FP], [FN, TP]]:')
print(matrix)
print(classification_report(y_test, test_predictions,
                            labels=[0, 1], target_names=['Ham', 'Spam'],
                            zero_division=0))
assert int(matrix.sum()) == len(test_text)
assert all(value >= 0 for value in (tn, fp, fn, tp))

## HTB's five new-message examples

Prediction-time preprocessing must match training: lowercase; keep `a-z`, whitespace, `$`, `!`; tokenize; remove the same stop words; stem; rejoin. The fitted pipeline then vectorizes the resulting strings automatically. We do **not** call `best_model.named_steps['vectorizer'].transform` and then feed that matrix to the whole pipeline, because that would vectorize twice. `predict_proba` columns follow `best_model.classes_`; locate the spam class by its label rather than assuming a column order. These handpicked messages illustrate behavior, not an unbiased evaluation set. They include fictional phishing URLs/phone numbers as *text examples*; nothing here visits them.

In [ ]:
new_messages = [
    "Congratulations! You've won a $1000 Walmart gift card. Go to http://bit.ly/1234 to claim now.",
    'Hey, are we still meeting up for lunch today?',
    'Urgent! Your account has been compromised. Verify your details here: www.fakebank.com/verify',
    'Reminder: Your appointment is scheduled for tomorrow at 10am.',
    'FREE entry in a weekly competition to win an iPad. Just text WIN to 80085 now!',
]
def preprocess_message(message):
    cleaned = re.sub(r'[^a-z\s$!]', '', message.lower())
    tokens = word_tokenize(cleaned)
    tokens = [word for word in tokens if word not in stop_words]
    return ' '.join(stemmer.stem(word) for word in tokens)

processed_messages = [preprocess_message(message) for message in new_messages]
predictions = best_model.predict(processed_messages)
probabilities = best_model.predict_proba(processed_messages)
spam_column = list(best_model.classes_).index(1)
ham_column = list(best_model.classes_).index(0)
for message, label, row in zip(new_messages, predictions, probabilities):
    print('Message:', message)
    print('Prediction:', 'Spam' if label == 1 else 'Not-Spam')
    print(f'Spam score: {row[spam_column]:.3f}; Ham score: {row[ham_column]:.3f}')
    print('-' * 35)
assert probabilities.shape == (len(new_messages), 2)
assert ((probabilities >= 0) & (probabilities <= 1)).all()
assert abs(probabilities.sum(axis=1) - 1).max() < 1e-10

## Demonstrate `joblib` save/load without leaving a model in the repo

`joblib.dump` serializes the **whole fitted pipeline**: vocabulary plus classifier parameters. `joblib.load` recreates it. Loading arbitrary downloaded model files is dangerous because Python object deserialization can execute code; load only files whose source you trust. The cell below loads only the file it just wrote inside a temporary directory in the current writable working folder, verifies identical predictions, then automatically removes that directory. We do not commit a trained model or treat serialization as model validation.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
import joblib

with TemporaryDirectory(prefix='htb_sms_study_', dir=Path.cwd()) as temporary_dir:
    model_path = Path(temporary_dir) / 'spam_detection_model.joblib'
    joblib.dump(best_model, model_path)
    # This is safe only because the file was created locally by this cell.
    reloaded_model = joblib.load(model_path)
    assert (reloaded_model.predict(processed_messages) == predictions).all()
print('Own-model serialization round-trip matched; temporary file removed')

Section 13 now has a trained pipeline, five-fold alpha selection on training data, one held-out test evaluation, five illustration predictions, and a temporary serialization round-trip. HTB's displayed confusion matrix and almost-0/1 example scores are **representative page output**, not guaranteed results of this downloaded archive, preprocessing version, or split. Naive Bayes probabilities are model estimates and may need calibration before being interpreted as real-world frequencies. No model file, flags, targets, or full dense feature matrix should be committed.

# Section 14 — Model Evaluation (Spam Detection)

HTB asks for a fitted `.joblib` model uploaded as the multipart form field `model` to its evaluator. In a Pwnbox the lesson uses `http://localhost:8000/api/upload`; from your own machine, spawn the HTB VM, connect the VPN, and replace `<VM-IP>` below with that VM's address. The server owns the hidden test set and decides whether the model qualifies. This account has already completed the question, so this cell does not need to resubmit or store its answer.

Our Section 13 held-out result was measured **before** final refitting. We can now use all available labeled messages to fit a final artifact with the selected alpha, but we cannot call the old held-out rows an untouched test set again. This is an upload workflow, not a new accuracy claim.

## Build the final fitted pipeline

`clone(best_model)` copies the chosen vectorizer/classifier *settings* without copying fitted counts or labels. The following `fit` learns a fresh vocabulary and Naive Bayes statistics from all available labeled rows. The fitted pipeline still expects the **Section 11 preprocessed text format** at inference time. Its `.predict` applies vectorization internally; do not feed it a count matrix. The earlier held-out test metrics remain the only local evaluation estimate.

In [ ]:
from sklearn.base import clone

final_model = clone(best_model)
final_model.fit(processed_sms['message'], y)
assert final_model.classes_.tolist() == [0, 1]
assert final_model.named_steps['classifier'].alpha == grid_search.best_params_['classifier__alpha']
assert final_model.named_steps['vectorizer'].get_feature_names_out().size > 0
print('Final model fitted on', len(processed_sms), 'labeled messages')

## Optional upload to the active HTB VM

Keep `RUN_HTB_UPLOAD = False` while studying or rerunning all cells. To submit to a newly spawned HTB VM, set the exact VM URL and flip the switch in this cell. `requests.post` sends the locally generated model file under the form key `model`; the response is printed for you to inspect. The temporary artifact is deleted afterward, including when the request fails. Only a challenge response that explicitly confirms success is evidence of passing. Never commit notebook output containing a flag.

In [ ]:
import requests

RUN_HTB_UPLOAD = False
HTB_EVALUATOR_URL = 'http://<VM-IP>:8000/api/upload'

if RUN_HTB_UPLOAD:
    if '<VM-IP>' in HTB_EVALUATOR_URL or not HTB_EVALUATOR_URL.endswith('/api/upload'):
        raise ValueError('Set the active HTB VM upload URL first.')
    with TemporaryDirectory(prefix='htb_sms_upload_', dir=Path.cwd()) as temporary_dir:
        model_path = Path(temporary_dir) / 'spam_detection_model.joblib'
        joblib.dump(final_model, model_path)
        with model_path.open('rb') as model_file:
            response = requests.post(
                HTB_EVALUATOR_URL, files={'model': model_file}, timeout=60)
        response.raise_for_status()
        try:
            result = response.json()
        except ValueError:
            print('Evaluator returned non-JSON text:', response.text[:500])
        else:
            print('Evaluator response:', result)
else:
    print('Upload disabled. Set the active HTB VM URL and opt in to submit.')

Section 14 stops at a validated, fitted final pipeline and a **disabled-by-default** HTB upload path. The portal performs its own hidden evaluation; this notebook does not claim to know those labels or generate a flag locally. The answer already shown by the completed HTB question remains outside Git. If a live upload is needed later, use only the active HTB VM address over the authorized connection and read the actual server response.